# Distance and Nearest Features

**Series:** Geoprocessing in Python (notebook 3 of 3)
**Time:** about 25 minutes

**In this notebook you will:**
- Measure the distance between features in meters
- Find the nearest supermarket to every tract with a nearest join
- Map distances and read them critically (straight line versus road distance)

**You should already know:** notebooks 01a and 01b.

> **Before you start:** click **File > Save a copy in Drive** so your changes are kept.
> Then run each cell in order by clicking it and pressing **Shift + Enter**.

## From zones to distances

A buffer gives a yes or no answer (inside or outside 1 mile). A **distance** keeps the detail: a tract 1.7 km
from a store and one 4 km away both fail the 1-mile test, but they are not equally far.

We measure from each tract's **centroid** (its center point) to the nearest supermarket. The centroid stands in
for where people live, which is a simplification we come back to at the end.

In [ ]:
import geopandas as gpd
import pandas as pd

stores = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/CaseStudies/main/data/grocery_stores.geojson")
city = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/CaseStudies/main/data/study_area.geojson")
tracts = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/Foundations/main/data/tippecanoe_tracts.geojson")
acs = pd.read_csv("https://raw.githubusercontent.com/SpatialTurn/Foundations/main/data/tippecanoe_acs_2024.csv", dtype={"GEOID": str})

# Work in meters: UTM zone 16N
stores = stores.to_crs(epsg=26916)
city = city.to_crs(epsg=26916)
tracts = tracts.to_crs(epsg=26916).merge(acs.drop(columns="name"), on="GEOID")
supermarkets = stores[stores["store_type"] == "Supermarket"]

centers = tracts.copy()
centers["geometry"] = tracts.centroid
centers = centers[centers.within(city.union_all())]     # tract centers inside the city limits
print(len(centers), "tract centers in the city")

## Step 1: Distance between two features

`a.distance(b)` returns the straight-line distance in the CRS units, here meters.

In [ ]:
purdue = centers[centers["name"] == "Census Tract 104"].geometry.iloc[0]
supermarkets.distance(purdue).sort_values().head(3).round()

## Step 2: Nearest join

Doing that for every tract by hand would be slow. `gpd.sjoin_nearest()` is a **spatial join** (Foundations
notebook 4c) that matches each feature to the closest feature in the other layer and can record the distance.

In [ ]:
nearest = gpd.sjoin_nearest(
    centers,
    supermarkets[["store_name", "geometry"]],
    distance_col="dist_m",
)
nearest[["name", "store_name", "dist_m"]].sort_values("dist_m", ascending=False).head(6).round()

If two stores are exactly the same distance away, a tract appears twice. Check that the row count did not grow.

In [ ]:
print(len(centers), "centers,", len(nearest), "rows after the join")
nearest = nearest[~nearest.index.duplicated()]
nearest["dist_m"].describe().round()

## Step 3: Map the distances

In [ ]:
tracts_dist = tracts.merge(nearest[["GEOID", "dist_m"]], on="GEOID")

ax = city.plot(color="whitesmoke", edgecolor="gray", figsize=(7, 7))
tracts_dist.clip(city).plot(ax=ax, column="dist_m", cmap="OrRd", scheme="NaturalBreaks", k=5,
                            edgecolor="white", legend=True,
                            legend_kwds={"title": "Meters to nearest\nsupermarket", "loc": "lower left",
                                         "fmt": "{:,.0f}"})
supermarkets.plot(ax=ax, color="black", markersize=10)
ax.set_title("Straight-line distance from tract center to nearest supermarket")
ax.set_axis_off()

`scheme="NaturalBreaks"` uses the **mapclassify** library (installed in Colab) to choose class breaks, as in the
Data Visualization module's map design episode.

## Reading the result critically

- **Straight line is not travel.** Roads, rivers and railroads make real trips longer. The Data Analysis
  Network Analysis episode and the Case Studies Grocery Access Part 2 episode measure along roads instead.
- **One point per tract.** A centroid can fall in a field or a parking lot. Population-weighted centers or
  smaller units (block groups, blocks) give fairer distances.
- **Edge effects.** A store just outside the city limits is invisible to this analysis.
- **Stores change.** OpenStreetMap data is a snapshot; check it against local knowledge.

## Try it yourself

The Foundations repo also has a layer of named parks from OpenStreetMap:
`https://raw.githubusercontent.com/SpatialTurn/Foundations/main/data/tippecanoe_parks.geojson`.

1. Read it and convert it to EPSG:26916.
2. Use `sjoin_nearest()` to find the nearest park to each tract center, with the distance.
3. Which tract center is farthest from a park, and how far is it?

*Hint:* Both layers have a `name` column, so give the join `lsuffix` and `rsuffix`. Distance to a polygon is measured to its nearest edge.

<details>
<summary><b>Show a solution</b></summary>

```python
parks = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/Foundations/main/data/tippecanoe_parks.geojson").to_crs(epsg=26916)
to_park = gpd.sjoin_nearest(centers, parks[["name", "geometry"]], distance_col="park_m",
                            lsuffix="tract", rsuffix="park")
to_park.sort_values("park_m", ascending=False)[["name_tract", "name_park", "park_m"]].head(3).round()
```

</details>

In [ ]:
# Write your answer here

## Recap

- `distance()` measures straight-line distance in the CRS units (project to meters first)
- `sjoin_nearest(..., distance_col=...)` matches every feature to its closest neighbor and records the distance
- Check row counts after a nearest join (ties add rows)
- Straight-line distance from a centroid is a simplification; network distance and smaller units are better

**Next:** the Raster Analysis episode, or apply these tools in the Case Studies module's park access case study.